In [ ]:
# ============================================================
# Single-file preprocessing pipeline for PhysioNet .psv patients
# Paste this whole file into one Jupyter notebook cell, or run it as a .py file.
#
# Pipeline order:
#   0. FullDataset baseline counts
#   1. Remove patients with less than 36 rows
#   2. Eliminate patients with all-NaN values in the feature list
#      NOTE: this follows your uploaded logic: delete the patient if ANY
#      required feature column is completely NaN for that patient.
#   3. Replace values outside operational limits with NaN
#   4. Keep/pad each patient to the last 48 hours
#
# After each process, the script shows:
#   - total patients left
#   - septic patients left
#   - nonseptic patients left
#   - unknown-label patients left
#   - removed/updated/skipped counts for that process
# ============================================================

import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None


# =========================
# User settings
# =========================

# Folder containing the .psv files you want to process.
DATA_DIR = Path("/Users/bryanbarrios/Desktop/MastersResearch/PredictiveModeling/Data/DatasetA")

# Column used to classify patients.
LABEL_COL = "SepsisLabel"

# Patient must have at least this many rows before the 48-hour formatting step.
MIN_ROWS = 36

# Final sequence length.
TIME_WINDOW = 48

# Features used for the all-NaN feature-column patient removal step.
FEATURES = ["HR", "O2Sat", "Temp", "SBP", "DBP", "Resp", "MAP"]

# Operating limits. Values outside these ranges are replaced with NaN.
LIMITS = {
    "HR":    (30, 220),
    "MAP":   (40, 160),
    "O2Sat": (70, 100),
    "Temp":  (32, 42),
    "SBP":   (40, 260),
    "DBP":   (20, 150),
    "Resp":  (4, 50),
}

# Set to True first if you want to preview counts without deleting/updating files.
DRY_RUN = False


# =========================
# Helper functions
# =========================

def get_psv_files(data_dir):
    """Return all .psv files in sorted order."""
    return sorted(Path(data_dir).glob("*.psv"))


def read_psv(fn):
    """Read one .psv file using the same NaN rules as your earlier scripts."""
    return pd.read_csv(fn, sep="|", na_values=["", "NaN", "nan"], keep_default_na=True)


def patient_status_from_df(df, label_col=LABEL_COL):
    """
    Classify one patient file.

    Septic:     patient has at least one SepsisLabel equal to 1
    Nonseptic:  patient has SepsisLabel values, but none equal to 1
    Unknown:    missing/empty SepsisLabel
    """
    if label_col not in df.columns or df.shape[0] == 0:
        return "Unknown"

    labels = pd.to_numeric(df[label_col], errors="coerce").dropna()

    if labels.empty:
        return "Unknown"
    elif (labels == 1).any():
        return "Septic"
    else:
        return "Nonseptic"


def cohort_counts(data_dir):
    """Count total, septic, nonseptic, unknown, and unreadable patients currently in DATA_DIR."""
    files = get_psv_files(data_dir)

    counts = {
        "Total Patients": len(files),
        "Septic": 0,
        "Nonseptic": 0,
        "Unknown Label": 0,
        "Unreadable Files": 0,
    }

    for fn in files:
        try:
            df = read_psv(fn)
            status = patient_status_from_df(df)
            if status == "Septic":
                counts["Septic"] += 1
            elif status == "Nonseptic":
                counts["Nonseptic"] += 1
            else:
                counts["Unknown Label"] += 1
        except Exception:
            counts["Unreadable Files"] += 1
            counts["Unknown Label"] += 1

    return counts


def add_summary_row(summary_rows, process_name, before_counts, after_counts,
                    removed=0, updated=0, skipped=0, notes=""):
    """Store one row for the summary table."""
    summary_rows.append({
        "Process": process_name,
        "Patients Before": before_counts.get("Total Patients", np.nan),
        "Patients After": after_counts.get("Total Patients", np.nan),
        "Removed": removed,
        "Updated": updated,
        "Skipped": skipped,
        "Septic After": after_counts.get("Septic", np.nan),
        "Nonseptic After": after_counts.get("Nonseptic", np.nan),
        "Unknown Label After": after_counts.get("Unknown Label", np.nan),
        "Unreadable Files After": after_counts.get("Unreadable Files", np.nan),
        "Notes": notes,
    })


def show_table(df, title=None):
    """Display nicely in Jupyter; otherwise print."""
    if title:
        print(f"\n{title}")
        print("-" * len(title))
    if display is not None:
        display(df)
    else:
        print(df.to_string(index=False))


# =========================
# Process 1: Drop patients with fewer than MIN_ROWS rows
# =========================

def drop_patients_with_less_than_min_rows(data_dir, min_rows=MIN_ROWS):
    """
    Delete a patient file if it has fewer than min_rows rows.
    This follows your drop_less_36_rows.py logic.
    """
    files = get_psv_files(data_dir)

    deleted = 0
    unreadable = 0

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            unreadable += 1
            continue

        if len(df) < min_rows:
            if not DRY_RUN:
                os.remove(fn)
            deleted += 1

    return {
        "removed": deleted,
        "updated": 0,
        "skipped": unreadable,
        "unreadable": unreadable,
    }


# =========================
# Process 2: Drop patients with any all-NaN required feature column
# =========================

def drop_patients_with_all_nan_feature(data_dir, features=FEATURES):
    """
    Delete a patient file if any required feature is completely NaN for that patient.

    Example: if HR is all NaN for patient_001.psv, then patient_001.psv is removed.
    This follows the logic from your drop_all_nan_cols.py file.
    """
    files = get_psv_files(data_dir)

    deleted = 0
    skipped_missing_features = 0
    unreadable = 0

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            unreadable += 1
            continue

        if any(c not in df.columns for c in features):
            skipped_missing_features += 1
            continue

        nan_cols = df[features].isna().all(axis=0).sum()

        if nan_cols >= 1:
            if not DRY_RUN:
                os.remove(fn)
            deleted += 1

    return {
        "removed": deleted,
        "updated": 0,
        "skipped": skipped_missing_features + unreadable,
        "skipped_missing_features": skipped_missing_features,
        "unreadable": unreadable,
    }


# =========================
# Process 3: Operating limits
# =========================

def apply_operating_limits(data_dir, limits=LIMITS):
    """
    Replace out-of-range values with NaN.
    This does not remove patients.
    """
    files = get_psv_files(data_dir)

    n_updated = 0
    n_bad = 0
    n_missing_limit_cols = 0
    total_replaced = {col: 0 for col in limits}

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            n_bad += 1
            continue

        changed_any = False
        missing_any = False

        for col, (lo, hi) in limits.items():
            if col not in df.columns:
                missing_any = True
                continue

            x = pd.to_numeric(df[col], errors="coerce")
            out = (x < lo) | (x > hi)

            if out.any():
                c = int(out.sum())
                total_replaced[col] += c
                df.loc[out, col] = np.nan
                changed_any = True

        if missing_any:
            n_missing_limit_cols += 1

        if changed_any:
            if not DRY_RUN:
                df.to_csv(fn, sep="|", index=False)
            n_updated += 1

    return {
        "removed": 0,
        "updated": n_updated,
        "skipped": n_bad,
        "missing_limit_cols": n_missing_limit_cols,
        "total_replaced": total_replaced,
    }


# =========================
# Process 4: Keep/pad to the last TIME_WINDOW hours
# =========================

def keep_or_pad_last_time_window(data_dir, time_window=TIME_WINDOW, label_col=LABEL_COL):
    """
    Make every patient file have exactly time_window rows.

    If rows > time_window:
        keep the last time_window rows.
    If rows < time_window:
        add NaN rows at the beginning and fill the padded SepsisLabel using the last observed label.
    If rows == time_window:
        reset the index and save.
    """
    files = get_psv_files(data_dir)

    updated = 0
    skipped = 0
    trimmed = 0
    padded = 0
    unchanged_length = 0

    for fn in files:
        try:
            df = read_psv(fn)
        except Exception:
            skipped += 1
            continue

        if df is None or df.shape[0] == 0 or label_col not in df.columns:
            skipped += 1
            continue

        n = len(df)

        if n > time_window:
            df_out = df.tail(time_window).reset_index(drop=True)
            trimmed += 1
        elif n < time_window:
            pad_n = time_window - n
            last_label = df[label_col].iloc[-1]

            pad = pd.DataFrame(np.nan, index=range(pad_n), columns=df.columns)
            pad[label_col] = last_label

            df_out = pd.concat([pad, df.reset_index(drop=True)], ignore_index=True)
            padded += 1
        else:
            df_out = df.reset_index(drop=True)
            unchanged_length += 1

        if not DRY_RUN:
            df_out.to_csv(fn, sep="|", index=False)
        updated += 1

    return {
        "removed": 0,
        "updated": updated,
        "skipped": skipped,
        "trimmed": trimmed,
        "padded": padded,
        "unchanged_length": unchanged_length,
    }


# =========================
# Run full pipeline in the requested order
# =========================

def run_pipeline(data_dir=DATA_DIR):
    data_dir = Path(data_dir)

    if not data_dir.exists():
        raise FileNotFoundError(f"DATA_DIR does not exist: {data_dir}")

    if len(get_psv_files(data_dir)) == 0:
        raise FileNotFoundError(f"No .psv files found in: {data_dir}")

    summary_rows = []

    # Process 0: FullDataset baseline counts
    full_counts = cohort_counts(data_dir)
    summary_rows.append({
        "Process": "0. FullDataset",
        "Patients Before": np.nan,
        "Patients After": full_counts["Total Patients"],
        "Removed": 0,
        "Updated": 0,
        "Skipped": 0,
        "Septic After": full_counts["Septic"],
        "Nonseptic After": full_counts["Nonseptic"],
        "Unknown Label After": full_counts["Unknown Label"],
        "Unreadable Files After": full_counts["Unreadable Files"],
        "Notes": "Original dataset before preprocessing",
    })

    # Process 1: Remove patients with less than 36 rows
    before = cohort_counts(data_dir)
    info_rows = drop_patients_with_less_than_min_rows(data_dir)
    after = cohort_counts(data_dir)
    add_summary_row(
        summary_rows,
        f"1. Remove patients with < {MIN_ROWS} rows",
        before,
        after,
        removed=info_rows["removed"],
        updated=info_rows["updated"],
        skipped=info_rows["skipped"],
        notes=f"Unreadable: {info_rows['unreadable']}",
    )

    # Process 2: Eliminate patients with all-NaN values in the feature list
    before = cohort_counts(data_dir)
    info_nan = drop_patients_with_all_nan_feature(data_dir)
    after = cohort_counts(data_dir)
    add_summary_row(
        summary_rows,
        "2. Eliminate patients with any all-NaN feature column",
        before,
        after,
        removed=info_nan["removed"],
        updated=info_nan["updated"],
        skipped=info_nan["skipped"],
        notes=f"Skipped missing required features: {info_nan['skipped_missing_features']}; unreadable: {info_nan['unreadable']}",
    )

    # Process 3: Replace values outside operational limits with NaN
    before = cohort_counts(data_dir)
    info_limits = apply_operating_limits(data_dir)
    after = cohort_counts(data_dir)
    replaced_short = ", ".join([f"{k}={v}" for k, v in info_limits["total_replaced"].items()])
    add_summary_row(
        summary_rows,
        "3. Replace values outside operational limits with NaN",
        before,
        after,
        removed=info_limits["removed"],
        updated=info_limits["updated"],
        skipped=info_limits["skipped"],
        notes=f"Values replaced: {replaced_short}. Files missing limit columns: {info_limits['missing_limit_cols']}",
    )

    # Process 4: Get patients' last 48 hours
    before = cohort_counts(data_dir)
    info_48 = keep_or_pad_last_time_window(data_dir)
    after = cohort_counts(data_dir)
    add_summary_row(
        summary_rows,
        f"4. Get patients' last {TIME_WINDOW} hours",
        before,
        after,
        removed=info_48["removed"],
        updated=info_48["updated"],
        skipped=info_48["skipped"],
        notes=f"Trimmed: {info_48['trimmed']}; padded: {info_48['padded']}; already {TIME_WINDOW} rows: {info_48['unchanged_length']}",
    )

    summary_df = pd.DataFrame(summary_rows)

    replacements_df = pd.DataFrame({
        "Feature": list(info_limits["total_replaced"].keys()),
        "Values Replaced With NaN": list(info_limits["total_replaced"].values()),
    })

    show_table(summary_df, "Patient count after each preprocessing process")
    show_table(replacements_df, "Operating-limit replacements by feature")

    return summary_df, replacements_df


# Run the pipeline.
summary_df, replacements_df = run_pipeline(DATA_DIR)
